In [1]:
import polars as pl
import polars_ols
from pathlib import Path

In [2]:
SCRIPT_DIR = Path(".").absolute().resolve()
DATA_DIR = (SCRIPT_DIR.parent / "data").absolute().resolve()
PROCESSED_DATA_DIR = (SCRIPT_DIR.parent / "processed").absolute().resolve()

print(f"{DATA_DIR=}")
print(f"{PROCESSED_DATA_DIR=}")
assert DATA_DIR.is_dir()
assert PROCESSED_DATA_DIR.is_dir()

DATA_DIR=PosixPath('/Users/Naot/kristal-thesis-not-mine/data')
PROCESSED_DATA_DIR=PosixPath('/Users/Naot/kristal-thesis-not-mine/processed')


In [3]:
price_df = pl.read_parquet(PROCESSED_DATA_DIR / "price.parquet")
price_df

ticker,date,close,volume,shares,mktcap
str,date,f64,i64,i64,i64
"""AAA""",2018-12-28,11536.08,1257250,171199976,2516639647200
"""AAM""",2018-12-28,8690.97,50,8035701,107678393400
"""ABR""",2018-12-28,3614.97,100,3000000,12300000000
"""ABT""",2018-12-28,26370.64,1350,11497257,462189731400
"""ACB""",2018-12-28,7864.52,2559551,1247165130,36916087848000
…,…,…,…,…,…
"""VIG""",2022-01-04,17700.0,1330695,34133300,604159410000
"""VIT""",2022-01-04,21428.46,15737,49999664,1169992137600
"""VLA""",2022-01-04,10379.33,55400,1080000,32400000000


In [4]:
icb_df = pl.read_parquet(PROCESSED_DATA_DIR / "icb.parquet")
icb_df

ticker,icb1,icb2,icb3,icb4,icb5
str,str,str,str,str,str
"""AAA""","""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa"""
"""AAM""","""Hàng Tiêu dùng""","""Thực phẩm và đồ uống""","""Sản xuất thực phẩm""","""Nuôi trồng nông & hải sản""","""Nuôi trồng thủy hải sản"""
"""AAT""","""Hàng Tiêu dùng""","""Hàng cá nhân & Gia dụng""","""Hàng cá nhân""","""Hàng May mặc""","""Hàng May mặc"""
"""ABR""","""Công nghiệp""","""Hàng & Dịch vụ Công nghiệp""","""Tư vấn & Hỗ trợ Kinh doanh""","""Tư vấn & Hỗ trợ KD""","""Tư vấn & Hỗ trợ KD"""
"""ABS""","""Hàng Tiêu dùng""","""Thực phẩm và đồ uống""","""Sản xuất thực phẩm""","""Nuôi trồng nông & hải sản""","""Trồng ngũ cốc, rau, trái cây &…"
…,…,…,…,…,…
"""VTV""","""Công nghiệp""","""Xây dựng và Vật liệu""","""Xây dựng và Vật liệu""","""Vật liệu xây dựng & Nội thất""","""Vật liệu xây dựng khác"""
"""VTZ""","""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa"""
"""WCS""","""Công nghiệp""","""Hàng & Dịch vụ Công nghiệp""","""Vận tải""","""Kho bãi, hậu cần và bảo dưỡng""","""Bến xe khách"""


In [5]:
cleaned_price_df = (
    price_df.join(
        icb_df,
        on="ticker",
        how="left",
        validate="m:1",
    )
    .filter(pl.col("icb2").is_not_null())
    .filter(
        ~pl.col("icb2").is_in(
            [
                "Dịch vụ tài chính",
                "Ngân hàng",
                "Bảo hiểm",
            ]
        )
    )
    .filter(~pl.col("ticker").is_in(["DSC", "MIC"]))
    .with_columns(
        pl.when(pl.col("mktcap") != 0)
        .then(pl.col("mktcap"))
        .otherwise(None)
        .alias("mktcap"),
        year=pl.col("date").dt.year(),
        week=pl.col("date").dt.week(),
        dayofweek=pl.col("date").dt.weekday(),
    )
    .filter(~pl.col("dayofweek").is_in([6, 7]))
    .filter(pl.col("year") >= 2010)
    .filter(pl.col("year") <= 2024)
)
cleaned_price_df

ticker,date,close,volume,shares,mktcap,icb1,icb2,icb3,icb4,icb5,year,week,dayofweek
str,date,f64,i64,i64,i64,str,str,str,str,str,i32,i8,i8
"""AAA""",2018-12-28,11536.08,1257250,171199976,2516639647200,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",2018,52,5
"""AAM""",2018-12-28,8690.97,50,8035701,107678393400,"""Hàng Tiêu dùng""","""Thực phẩm và đồ uống""","""Sản xuất thực phẩm""","""Nuôi trồng nông & hải sản""","""Nuôi trồng thủy hải sản""",2018,52,5
"""ABR""",2018-12-28,3614.97,100,3000000,12300000000,"""Công nghiệp""","""Hàng & Dịch vụ Công nghiệp""","""Tư vấn & Hỗ trợ Kinh doanh""","""Tư vấn & Hỗ trợ KD""","""Tư vấn & Hỗ trợ KD""",2018,52,5
"""ABT""",2018-12-28,26370.64,1350,11497257,462189731400,"""Hàng Tiêu dùng""","""Thực phẩm và đồ uống""","""Sản xuất thực phẩm""","""Nuôi trồng nông & hải sản""","""Nuôi trồng thủy hải sản""",2018,52,5
"""ACC""",2018-12-28,5718.26,40,10000000,218000000000,"""Công nghiệp""","""Xây dựng và Vật liệu""","""Xây dựng và Vật liệu""","""Vật liệu xây dựng & Nội thất""","""Sản xuất bê tông""",2018,52,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VIF""",2022-01-04,17585.17,0,350000000,7105000000000,"""Nguyên vật liệu""","""Tài nguyên Cơ bản""","""Lâm nghiệp và Giấy""","""Lâm sản và Chế biến gỗ""","""Lâm sản và Chế biến gỗ""",2022,1,2
"""VIT""",2022-01-04,21428.46,15737,49999664,1169992137600,"""Công nghiệp""","""Xây dựng và Vật liệu""","""Xây dựng và Vật liệu""","""Vật liệu xây dựng & Nội thất""","""Sản xuất gạch ốp lát & Vật liệ…",2022,1,2
"""VLA""",2022-01-04,10379.33,55400,1080000,32400000000,"""Công nghệ Thông tin""","""Công nghệ Thông tin""","""Phần mềm & Dịch vụ Máy tính""","""Phần mềm""","""Phần mềm""",2022,1,2


In [6]:
wret_df = (
    cleaned_price_df.with_columns(pl.col("date").cast(pl.Date))
    .sort(["ticker", "date"])
    .upsample(time_column="date", every="1d", group_by="ticker", maintain_order=True)
    .with_columns(
        dret=(pl.col("close") / pl.col("close").shift(1)).log().over("ticker")
    )
    .filter(pl.col("close").is_not_null() | pl.col("dret").is_not_null())
    .with_columns(
        year=pl.col("date").dt.year(),
        week=pl.col("date").dt.week(),
        mktcap=pl.when(pl.col("mktcap") != 0).then(pl.col("mktcap")).otherwise(None),
    )
    .group_by(["ticker", "year", "week"])
    .agg(
        [
            pl.col("dret").sum(),
            pl.col("mktcap").last(ignore_nulls=True),
            pl.col("icb1").last(ignore_nulls=True),
            pl.col("icb2").last(ignore_nulls=True),
            pl.col("icb3").last(ignore_nulls=True),
            pl.col("icb4").last(ignore_nulls=True),
            pl.col("icb5").last(ignore_nulls=True),
        ]
    )
    # 7. Final Weekly Return and Outlier Filter
    .with_columns(wret=(pl.col("dret").exp() - 1))
    .filter(pl.col("wret").abs() < (pow(1.1, 5) - 1))
)
wret_df

ticker,year,week,dret,mktcap,icb1,icb2,icb3,icb4,icb5,wret
str,i32,i8,f64,i64,str,str,str,str,str,f64
"""VE1""",2018,37,0.019231,62278440000,"""Công nghiệp""","""Xây dựng và Vật liệu""","""Xây dựng và Vật liệu""","""Xây dựng""","""Xây dựng""",0.019417
"""CLC""",2024,27,0.013699,1155754410300,"""Hàng Tiêu dùng""","""Hàng cá nhân & Gia dụng""","""Thuốc lá""","""Thuốc lá""","""Thuốc lá""",0.013793
"""KTS""",2018,12,-0.1494,106977000000,"""Hàng Tiêu dùng""","""Thực phẩm và đồ uống""","""Sản xuất thực phẩm""","""Thực phẩm""","""Đường""",-0.138776
"""S4A""",2020,18,0.0,1160500000000,"""Tiện ích Cộng đồng""","""Điện, nước & xăng dầu khí đốt""","""Sản xuất & Phân phối Điện""","""Sản xuất & Phân phối Điện""","""Sản xuất & Phân phối Điện""",0.0
"""VFG""",2022,14,-0.04512,2085646160000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Sản phẩm hóa dầu, Nông dược & …","""Thuốc trừ sâu""",-0.044118
…,…,…,…,…,…,…,…,…,…,…
"""BWE""",2018,19,-0.045053,3255000000000,"""Tiện ích Cộng đồng""","""Điện, nước & xăng dầu khí đốt""","""Nước & Khí đốt""","""Nước""","""Nước""",-0.044053
"""SMB""",2012,49,0.113191,250711843200,"""Hàng Tiêu dùng""","""Thực phẩm và đồ uống""","""Bia và đồ uống""","""Sản xuất bia""","""Sản xuất bia""",0.119845
"""HMH""",2024,31,0.04226,186290881500,"""Công nghiệp""","""Hàng & Dịch vụ Công nghiệp""","""Vận tải""","""Dịch vụ vận tải""","""Vận tải hàng khô""",0.043165


In [7]:
vwmktret_df = (
    wret_df.sort(["ticker", "year", "week"])
    .with_columns(
        weekid=pl.int_range(0, pl.len()).over("ticker"),
        totalmktcap=pl.col("mktcap").sum().over(["year", "week"]),
    )
    .with_columns(mktcap_weighted=pl.col("mktcap") / pl.col("totalmktcap"))
    .pipe(
        lambda x: x.join(
            x.select(
                [
                    pl.col("ticker"),
                    (pl.col("weekid") + 1).alias("weekid"),
                    pl.col("mktcap_weighted").alias("mktcap_weighted_L1"),
                ]
            ),
            on=["ticker", "weekid"],
            how="left",
        )
    )
    .with_columns(wret_weighted=pl.col("mktcap_weighted_L1") * pl.col("wret"))
    .with_columns(vwmktret=pl.col("wret_weighted").sum().over(["year", "week"]))
)
vwmktret_df

ticker,year,week,dret,mktcap,icb1,icb2,icb3,icb4,icb5,wret,weekid,totalmktcap,mktcap_weighted,mktcap_weighted_L1,wret_weighted,vwmktret
str,i32,i8,f64,i64,str,str,str,str,str,f64,i64,i64,f64,f64,f64,f64
"""AAA""",2010,28,-0.068421,461340000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.066132,0,367582941393100,0.001255,null,null,0.007874
"""AAA""",2010,29,-0.008565,460350000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.008529,1,364504407104800,0.001263,0.001255,-0.000011,-0.011679
"""AAA""",2010,30,0.056918,483120000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.058569,2,368471467333800,0.001311,0.001263,0.000074,-0.003428
"""AAA""",2010,31,-0.004073,485100000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.004065,3,361565438814300,0.001342,0.001311,-0.000005,-0.013798
"""AAA""",2010,32,-0.091767,432630000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.087682,4,341645725542300,0.001266,0.001342,-0.000118,-0.038262
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YEG""",2024,48,0.041577,1513866066700,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.042453,333,3076030810809110,0.000492,0.000485,0.000021,0.010507
"""YEG""",2024,49,0.183066,1842669556300,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.200893,334,3122364078266650,0.00059,0.000492,0.000099,0.016263
"""YEG""",2024,50,0.024098,2013921373800,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.024391,335,3100006804702970,0.00065,0.00059,0.000014,-0.008959


In [8]:
vwindret_df = (
    vwmktret_df.with_columns(
        total_mktcap_ind=pl.col("mktcap").sum().over(["icb1", "year", "week"])
    )
    .with_columns(ind_weight=pl.col("mktcap") / pl.col("total_mktcap_ind"))
    .pipe(
        lambda x: x.join(
            x.select(
                [
                    pl.col("ticker"),
                    (pl.col("weekid") + 1).alias("weekid"),
                    pl.col("ind_weight").alias("ind_weight_L1"),
                ]
            ),
            on=["ticker", "weekid"],
            how="left",
        )
    )
    .with_columns(ind_wret_weighted=pl.col("ind_weight_L1") * pl.col("wret"))
    .with_columns(
        vwindret=pl.col("ind_wret_weighted").sum().over(["icb1", "year", "week"])
    )
)
vwindret_df

ticker,year,week,dret,mktcap,icb1,icb2,icb3,icb4,icb5,wret,weekid,totalmktcap,mktcap_weighted,mktcap_weighted_L1,wret_weighted,vwmktret,total_mktcap_ind,ind_weight,ind_weight_L1,ind_wret_weighted,vwindret
str,i32,i8,f64,i64,str,str,str,str,str,f64,i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64
"""AAA""",2010,28,-0.068421,461340000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.066132,0,367582941393100,0.001255,null,null,0.007874,51222511130200,0.009007,null,null,0.011574
"""AAA""",2010,29,-0.008565,460350000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.008529,1,364504407104800,0.001263,0.001255,-0.000011,-0.011679,50790276928500,0.009064,0.009007,-0.000077,-0.00809
"""AAA""",2010,30,0.056918,483120000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.058569,2,368471467333800,0.001311,0.001263,0.000074,-0.003428,50168588971500,0.00963,0.009064,0.000531,-0.012907
"""AAA""",2010,31,-0.004073,485100000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.004065,3,361565438814300,0.001342,0.001311,-0.000005,-0.013798,49937355851500,0.009714,0.00963,-0.000039,0.003191
"""AAA""",2010,32,-0.091767,432630000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.087682,4,341645725542300,0.001266,0.001342,-0.000118,-0.038262,46369539505400,0.00933,0.009714,-0.000852,-0.040773
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YEG""",2024,48,0.041577,1513866066700,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.042453,333,3076030810809110,0.000492,0.000485,0.000021,0.010507,269273482539460,0.005622,0.005585,0.000237,0.014501
"""YEG""",2024,49,0.183066,1842669556300,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.200893,334,3122364078266650,0.00059,0.000492,0.000099,0.016263,268268050597380,0.006869,0.005622,0.001129,-0.007632
"""YEG""",2024,50,0.024098,2013921373800,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.024391,335,3100006804702970,0.00065,0.00059,0.000014,-0.008959,267917880714000,0.007517,0.006869,0.000168,-0.006927


In [13]:
ols_df = (
    vwindret_df.sort(
        ["ticker", "year", "week"]
    )  # <--- Optional but recommended safety net
    .with_columns(
        vwmktret_L1=pl.col("vwmktret").shift(1).over("ticker"),
        vwmktret_F1=pl.col("vwmktret").shift(-1).over("ticker"),
        vwindret_L1=pl.col("vwindret").shift(1).over("ticker"),
        vwindret_F1=pl.col("vwindret").shift(-1).over("ticker"),
    )
    .with_columns(
        ols_res=pl.col("wret")
        .least_squares.rolling_ols(
            pl.col("vwmktret_L1"),
            pl.col("vwmktret"),
            pl.col("vwmktret_F1"),
            pl.col("vwindret_L1"),
            pl.col("vwindret"),
            pl.col("vwindret_F1"),
            window_size=53,
            min_periods=26,
            add_intercept=True,
            mode="coefficients",
        )
        .over("ticker")
    )
    .filter(pl.col("ols_res").is_not_null())
    .with_columns(
        b_vwmktret_L1=pl.col("ols_res").struct.field("vwmktret_L1"),
        b_vwmktret=pl.col("ols_res").struct.field("vwmktret"),
        b_vwmktret_F1=pl.col("ols_res").struct.field("vwmktret_F1"),
        b_vwindret_L1=pl.col("ols_res").struct.field("vwindret_L1"),
        b_vwindret=pl.col("ols_res").struct.field("vwindret"),
        b_vwindret_F1=pl.col("ols_res").struct.field("vwindret_F1"),
        b_cons=pl.col("ols_res").struct.field("const"),
    )
    .with_columns(
        e=pl.col("wret")
        - (
            pl.col("b_vwmktret_L1") * pl.col("vwmktret_L1")
            + pl.col("b_vwmktret") * pl.col("vwmktret")
            + pl.col("b_vwmktret_F1") * pl.col("vwmktret_F1")
            + pl.col("b_vwindret_L1") * pl.col("vwindret_L1")
            + pl.col("b_vwindret") * pl.col("vwindret")
            + pl.col("b_vwindret_F1") * pl.col("vwindret_F1")
            + pl.col("b_cons")
        )
    )
    .with_columns(W=pl.col("e").log1p())
    .drop_nulls(subset=["W"])
    .drop(["e", "ols_res"])
)
ols_df

ticker,year,week,dret,mktcap,icb1,icb2,icb3,icb4,icb5,wret,weekid,totalmktcap,mktcap_weighted,mktcap_weighted_L1,wret_weighted,vwmktret,total_mktcap_ind,ind_weight,ind_weight_L1,ind_wret_weighted,vwindret,vwmktret_L1,vwmktret_F1,vwindret_L1,vwindret_F1,b_vwmktret_L1,b_vwmktret,b_vwmktret_F1,b_vwindret_L1,b_vwindret,b_vwindret_F1,b_cons,W
str,i32,i8,f64,i64,str,str,str,str,str,f64,i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAA""",2011,2,0.008733,341550000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.008771,26,388283939361400,0.00088,0.000854,0.000007,0.004718,50982239216800,0.006699,0.006524,0.000057,0.001201,-0.016609,0.028262,-0.019265,0.053787,-1.973624,-1.885019,0.855566,1.323383,3.427124,0.934459,0.002225,-0.073005
"""AAA""",2011,3,0.069376,369270000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.071839,27,403986235405600,0.000914,0.00088,0.000063,0.028262,53885162758400,0.006853,0.006699,0.000481,0.053787,0.004718,0.013442,0.001201,0.00434,-1.882128,-1.470484,0.921217,1.094213,2.828251,0.760462,-0.000756,-0.047181
"""AAA""",2011,4,-1.0061e-16,345510000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-1.1102e-16,28,406096903897800,0.000851,0.000914,-1.0148e-19,0.013442,53499979158000,0.006458,0.006853,-7.6083e-19,0.00434,0.028262,-0.003575,0.053787,-0.010383,-1.951224,-1.46537,0.897116,1.191609,2.846664,0.783306,-0.000275,0.009961
"""AAA""",2011,6,0.002912,340560000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.002916,29,413330163504700,0.000824,0.000851,0.000002,-0.003575,53984564337000,0.006308,0.006458,0.000019,-0.010383,0.013442,-0.028048,0.00434,-0.043454,-1.476038,-1.118815,1.370132,0.683655,2.344612,0.14878,0.002952,0.078881
"""AAA""",2011,7,-0.029327,332640000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.028902,30,401053938083400,0.000829,0.000824,-0.000024,-0.028048,50792453178200,0.006549,0.006308,-0.000182,-0.043454,-0.003575,-0.013652,-0.010383,-0.017349,-1.296937,-0.907373,1.561667,0.430483,1.99985,-0.088366,0.004303,0.046757
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YEG""",2024,47,-0.045463,1472765630500,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",-0.044445,332,3035066060732380,0.000485,0.000509,-0.000023,0.004563,263716180585730,0.005585,0.005926,-0.000263,0.023943,-0.021796,0.010507,-0.026869,0.014501,-0.722776,0.261162,0.871549,0.834905,0.01769,-0.652802,0.002136,-0.042082
"""YEG""",2024,48,0.041577,1513866066700,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.042453,333,3076030810809110,0.000492,0.000485,0.000021,0.010507,269273482539460,0.005622,0.005585,0.000237,0.014501,0.004563,0.016263,0.023943,-0.007632,-0.698473,0.270491,0.835115,0.827715,0.032733,-0.637421,0.001512,0.002543
"""YEG""",2024,49,0.183066,1842669556300,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.200893,334,3122364078266650,0.00059,0.000492,0.000099,0.016263,268268050597380,0.006869,0.005622,0.001129,-0.007632,0.010507,-0.008959,0.014501,-0.006927,-0.503626,0.804828,0.765943,0.819764,-0.306844,-0.671661,0.005374,0.161866


In [16]:
df_stats = ols_df.sort(["ticker", "year", "week"]).with_columns(
    RET=pl.col("W").mean().over(["ticker", "year"]),
    STDEV=pl.col("W").std().over(["ticker", "year"]),
)
df_stats

ticker,year,week,dret,mktcap,icb1,icb2,icb3,icb4,icb5,wret,weekid,totalmktcap,mktcap_weighted,mktcap_weighted_L1,wret_weighted,vwmktret,total_mktcap_ind,ind_weight,ind_weight_L1,ind_wret_weighted,vwindret,vwmktret_L1,vwmktret_F1,vwindret_L1,vwindret_F1,b_vwmktret_L1,b_vwmktret,b_vwmktret_F1,b_vwindret_L1,b_vwindret,b_vwindret_F1,b_cons,W,RET,STDEV
str,i32,i8,f64,i64,str,str,str,str,str,f64,i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AAA""",2011,2,0.008733,341550000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.008771,26,388283939361400,0.00088,0.000854,0.000007,0.004718,50982239216800,0.006699,0.006524,0.000057,0.001201,-0.016609,0.028262,-0.019265,0.053787,-1.973624,-1.885019,0.855566,1.323383,3.427124,0.934459,0.002225,-0.073005,-0.002458,0.052178
"""AAA""",2011,3,0.069376,369270000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.071839,27,403986235405600,0.000914,0.00088,0.000063,0.028262,53885162758400,0.006853,0.006699,0.000481,0.053787,0.004718,0.013442,0.001201,0.00434,-1.882128,-1.470484,0.921217,1.094213,2.828251,0.760462,-0.000756,-0.047181,-0.002458,0.052178
"""AAA""",2011,4,-1.0061e-16,345510000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-1.1102e-16,28,406096903897800,0.000851,0.000914,-1.0148e-19,0.013442,53499979158000,0.006458,0.006853,-7.6083e-19,0.00434,0.028262,-0.003575,0.053787,-0.010383,-1.951224,-1.46537,0.897116,1.191609,2.846664,0.783306,-0.000275,0.009961,-0.002458,0.052178
"""AAA""",2011,6,0.002912,340560000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.002916,29,413330163504700,0.000824,0.000851,0.000002,-0.003575,53984564337000,0.006308,0.006458,0.000019,-0.010383,0.013442,-0.028048,0.00434,-0.043454,-1.476038,-1.118815,1.370132,0.683655,2.344612,0.14878,0.002952,0.078881,-0.002458,0.052178
"""AAA""",2011,7,-0.029327,332640000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.028902,30,401053938083400,0.000829,0.000824,-0.000024,-0.028048,50792453178200,0.006549,0.006308,-0.000182,-0.043454,-0.003575,-0.013652,-0.010383,-0.017349,-1.296937,-0.907373,1.561667,0.430483,1.99985,-0.088366,0.004303,0.046757,-0.002458,0.052178
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YEG""",2024,47,-0.045463,1472765630500,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",-0.044445,332,3035066060732380,0.000485,0.000509,-0.000023,0.004563,263716180585730,0.005585,0.005926,-0.000263,0.023943,-0.021796,0.010507,-0.026869,0.014501,-0.722776,0.261162,0.871549,0.834905,0.01769,-0.652802,0.002136,-0.042082,0.006286,0.058895
"""YEG""",2024,48,0.041577,1513866066700,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.042453,333,3076030810809110,0.000492,0.000485,0.000021,0.010507,269273482539460,0.005622,0.005585,0.000237,0.014501,0.004563,0.016263,0.023943,-0.007632,-0.698473,0.270491,0.835115,0.827715,0.032733,-0.637421,0.001512,0.002543,0.006286,0.058895
"""YEG""",2024,49,0.183066,1842669556300,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.200893,334,3122364078266650,0.00059,0.000492,0.000099,0.016263,268268050597380,0.006869,0.005622,0.001129,-0.007632,0.010507,-0.008959,0.014501,-0.006927,-0.503626,0.804828,0.765943,0.819764,-0.306844,-0.671661,0.005374,0.161866,0.006286,0.058895


In [ ]:
df_crash = (
    df_stats
    # 1. Sort the data
    .sort(["ticker", "year", "week"])
    # 2. Pre-collapse variable generation
    .with_columns(
        W2=pl.col("W").pow(2),
        W3=pl.col("W").pow(3),
        # In Stata, 'if' leaves the value as missing (.) when false. We use 'None' to mimic this.
        W_up2=pl.when(pl.col("W") > pl.col("RET"))
        .then(pl.col("W").pow(2))
        .otherwise(None),
        W_down2=pl.when(pl.col("W") < pl.col("RET"))
        .then(pl.col("W").pow(2))
        .otherwise(None),
        # Flag extreme negative returns (>= 3.09 std devs)
        DCRASH=pl.when(pl.col("W").abs() >= 3.09 * pl.col("STDEV"))
        .then(1)
        .otherwise(0),
    )
    # 3. Collapse (Aggregate to the Year level)
    .group_by(["ticker", "year"], maintain_order=True)
    .agg(
        [
            # (count) maps to .count() because Polars' count() correctly ignores nulls
            pl.col("W").count().alias("W"),  # Note: W is now the 'N' count!
            pl.col("W_up2").count().alias("n_up"),
            pl.col("W_down2").count().alias("n_down"),
            # (sum)
            pl.col("W2").sum(),
            pl.col("W3").sum(),
            pl.col("W_up2").sum(),
            pl.col("W_down2").sum(),
            pl.col("DCRASH").sum(),
            # (lastnm)
            pl.col("mktcap").last(ignore_nulls=True),
            pl.col("icb1").last(ignore_nulls=True),
            pl.col("icb2").last(ignore_nulls=True),
            pl.col("icb3").last(ignore_nulls=True),
            pl.col("icb4").last(ignore_nulls=True),
            pl.col("icb5").last(ignore_nulls=True),
            pl.col("RET").last(ignore_nulls=True),
            pl.col("STDEV").last(ignore_nulls=True),
        ]
    )
    # 4. Post-collapse Skewness and Volatility math
    .with_columns(
        # Stata evaluates (3/2) as 1.5. Using .pow(1.5) is the strict mathematical equivalent.
        NCSKEW1=pl.col("W") * (pl.col("W") - 1).pow(1.5) * pl.col("W3"),
        NCSKEW2=(pl.col("W") - 1) * (pl.col("W") - 2) * pl.col("W2").pow(1.5),
    )
    .with_columns(
        NCSKEW=-pl.col("NCSKEW1") / pl.col("NCSKEW2"),
        # Natural log is .log() in Polars
        DUVOL=(
            ((pl.col("n_up") - 1) * pl.col("W_down2"))
            / ((pl.col("n_down") - 1) * pl.col("W_up2"))
        ).log(),
        # replace DCRASH=1 if DCRASH>0 (Since we summed it, any positive value means at least 1 crash week)
        DCRASH=pl.when(pl.col("DCRASH") > 0).then(1).otherwise(0),
    )
    # (Optional) Drop intermediate calculation columns if you want a clean final dataset
    # .drop(["W2", "W3", "W_up2", "W_down2", "NCSKEW1", "NCSKEW2", "n_up", "n_down"])
)

df_crash

ticker,year,W,n_up,n_down,W2,W3,W_up2,W_down2,DCRASH,mktcap,icb1,icb2,icb3,icb4,icb5,RET,STDEV,NCSKEW1,NCSKEW2,NCSKEW,DUVOL
str,i32,u32,u32,u32,f64,f64,f64,f64,i32,i64,str,str,str,str,str,f64,f64,f64,f64,f64,f64
"""AAA""",2011,50,24,26,0.133708,-0.002052,0.059773,0.073935,0,111870000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.002458,0.052178,-35.197463,114.993744,0.306082,0.129243
"""AAA""",2012,51,28,23,0.093612,-0.001185,0.039849,0.053763,0,273240000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.002088,0.043218,-21.370587,70.172236,0.304545,0.504287
"""AAA""",2013,51,25,26,0.03597,0.000833,0.023068,0.012902,0,350460000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.002398,0.026712,15.025375,16.713754,-0.898983,-0.621924
"""AAA""",2014,52,29,23,0.093701,-0.000955,0.044286,0.049415,0,534600000000,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",0.000122,0.042863,-18.09335,73.140051,0.24738,0.350751
"""AAA""",2015,51,28,23,0.059844,-0.000906,0.024209,0.035635,0,608849852400,"""Nguyên vật liệu""","""Hóa chất""","""Hóa chất""","""Nhựa, cao su & sợi""","""Nhựa""",-0.002558,0.034499,-16.32759,35.867244,0.455223,0.591367
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""YEG""",2020,53,25,28,0.231183,0.027195,0.182505,0.048678,1,1442006524800,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",0.013729,0.065221,540.473144,294.786438,-1.83344,-1.439329
"""YEG""",2021,52,29,23,0.213792,-0.013648,0.075897,0.137895,1,797639184000,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",-0.004321,0.064599,-258.476848,252.074178,1.0254,0.838275
"""YEG""",2022,51,21,30,0.245792,0.007634,0.136256,0.109536,0,278704514880,"""Dịch vụ Tiêu dùng""","""Truyền thông""","""Truyền thông""","""Giải trí & Truyền thông""","""Giải trí & Truyền thông""",-0.004718,0.069951,137.644701,298.54999,-0.461044,-0.589845
